# Week 5-3 — Multi-Query Retrieval

**목적**: 사용자의 질문 표현과 문서의 서술 방식이 달라 검색이 실패하는 경우를 줄인다. 하나의 질문을 LLM으로 여러 표현으로 변형해 각각 검색한 뒤, 결과를 합쳐 최종 문서를 고른다.

**가설**: 5-2 Error Case에서 관찰된 실패 유형 — BI-RADS 밀도 4단계, 위험 요인 나열처럼 **답이 여러 항목에 분산된 질문** — 은 단일 질문으로는 top-5에 다 담기 어렵다. 질문을 여러 각도로 변형하면 분산된 근거를 더 넓게 회수할 수 있을 것이다.

**구성**
- base retriever: 5-2에서 확정한 **Hybrid + Rerank**
- R5_multiquery: 원 질문 + LLM 생성 변형 질문 3개 → 각각 Hybrid 검색 → RRF 융합 → Rerank → top-5
- R1~R4는 기존 결과를 재사용하고, R5만 신규 채점해 5구성 비교표를 완성한다.
- judge: gpt-4o-mini (5주차 통일)

---
## 1. 설정

In [1]:
from pathlib import Path
import os, json, re
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_EVAL = PROJECT_ROOT / "data" / "eval"
VECTOR_ROOT = PROJECT_ROOT / "data" / "vector_store"

load_dotenv(PROJECT_ROOT / ".env"); load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 없음"

EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
GEN_MODEL = "gpt-4o-mini"
JUDGE_MODEL = "gpt-4o-mini"
QUERY_GEN_MODEL = "gpt-4o-mini"   # 질문 변형 생성
TOP_K = 5
FETCH_K = 20
RRF_K = 60
N_VARIANTS = 3                     # 원 질문 외 추가 변형 수

CHUNK_SIZE_BY_LANG    = {"ko": 540, "en": 620, "unknown": 580}
CHUNK_OVERLAP_BY_LANG = {"ko": 80,  "en": 90,  "unknown": 85}

CUT_FROM_PAGE = {
    "esmo_breast_cancer_patient_guide_korean.pdf": 60,
    "ncc_breast_cancer_screening_guideline_2015.pdf": 109,
    "nccn_metastatic_breast_cancer_patient.pdf": 63,
}
KBCS_REF_PAGES = set([
    109, 111, 115, 117, 121, 123, 124, 125, 126, 127, 129, 130, 131, 132, 135,
    137, 139, 141, 172, 173, 174, 223, 232, 233, 234, 235, 236, 237, 238, 240,
    241, 242, 243, 244, 245, 247, 248, 249, 250, 251, 252,
    50, 51, 53, 56, 100, 101, 103, 104, 108, 110, 112, 113, 114, 116, 118,
    119, 120, 122, 128, 133, 134, 136, 138, 140, 163, 164, 167, 168, 169, 170,
    171, 221, 222, 224, 227, 228, 229, 230, 231, 239, 246,
])
KBCS_NAME = "kbcs_korean_breast_cancer_guideline_2023.pdf"
print("설정 완료")

설정 완료


---
## 2. base 파이프라인 재구성 (P1 chunk → Dense/BM25/Hybrid → Rerank)

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

manifest_path = DATA_RAW / "metadata" / "manifest.json"
meta_lookup = {}
if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        meta_lookup = {m["filename"]: m for m in json.load(f) if m.get("downloaded")}

def guess_lang(text):
    return "ko" if len(re.findall(r"[\uac00-\ud7a3]", text)) > 20 else "en"

def load_docs_p1():
    out = []
    for pdf in sorted((DATA_RAW / "pdf").rglob("*.pdf")):
        cut = CUT_FROM_PAGE.get(pdf.name)
        for d in PyMuPDFLoader(str(pdf)).load():
            pg = d.metadata.get("page", 0)
            if cut is not None and pg >= cut: continue
            if pdf.name == KBCS_NAME and pg in KBCS_REF_PAGES: continue
            extra = meta_lookup.get(pdf.name, {})
            d.metadata.update({"filename": pdf.name, "org": extra.get("org", pdf.parent.name),
                "title": extra.get("title", pdf.stem),
                "language": extra.get("language", guess_lang(d.page_content)), "page": pg})
            out.append(d)
    return out

SEPARATORS = ["\n\n", "\n", ". ", " ", ""]
def split_docs(docs):
    out = []
    for lang in set(d.metadata.get("language", "unknown") for d in docs):
        size = CHUNK_SIZE_BY_LANG.get(lang, 580); ov = CHUNK_OVERLAP_BY_LANG.get(lang, 85)
        sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=ov,
                                            separators=SEPARATORS, length_function=len)
        out.extend(sp.split_documents([d for d in docs if d.metadata.get("language") == lang]))
    return out

chunks = split_docs(load_docs_p1())
print(f"P1 chunk: {len(chunks)}개")

P1 chunk: 2240개


In [3]:
import os as _os
from huggingface_hub import snapshot_download
_os.environ.pop("HF_HUB_OFFLINE", None); _os.environ.pop("TRANSFORMERS_OFFLINE", None)
_os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

try:
    model_dir = snapshot_download(repo_id=EMBEDDING_MODEL, local_files_only=True)
except Exception:
    model_dir = snapshot_download(repo_id=EMBEDDING_MODEL, local_files_only=False, max_workers=1)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from rank_bm25 import BM25Okapi
from kiwipiepy import Kiwi
from sentence_transformers import CrossEncoder
from tqdm import tqdm

embeddings = HuggingFaceEmbeddings(model_name=model_dir,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16})

vs = Chroma(collection_name="breast_rag_week5pre_P1_ref_removed",
            embedding_function=embeddings,
            persist_directory=str(VECTOR_ROOT / "week5pre_P1_ref_removed"))
assert vs._collection.count() > 0, "P1 컬렉션 없음 — 5-0 먼저 실행"

kiwi = Kiwi()
def tokenize(text):
    return [t.form.lower() for t in kiwi.tokenize(text)
            if t.tag[0] in ("N", "V", "S") or t.tag in ("SL", "SN", "XR")]

corpus_tokens = [tokenize(c.page_content) for c in tqdm(chunks, desc="BM25 토큰화")]
bm25 = BM25Okapi(corpus_tokens)
reranker = CrossEncoder(RERANK_MODEL, max_length=512, device="cpu")

def dense_search(query, k=TOP_K):
    return vs.similarity_search(query, k=k)

def bm25_search(query, k=TOP_K):
    scores = bm25.get_scores(tokenize(query))
    idx = sorted(range(len(scores)), key=lambda i: -scores[i])[:k]
    return [chunks[i] for i in idx]

def _key(doc):
    return (doc.metadata.get("filename"), doc.metadata.get("page"), doc.page_content[:80])

def _rrf_merge(doc_lists, k):
    """여러 검색 결과 리스트를 RRF로 융합"""
    scores, registry = {}, {}
    for docs in doc_lists:
        for rank, d in enumerate(docs):
            kk = _key(d); registry[kk] = d
            scores[kk] = scores.get(kk, 0) + 1.0 / (RRF_K + rank + 1)
    top = sorted(scores, key=lambda kk: -scores[kk])[:k]
    return [registry[kk] for kk in top]

def hybrid_search(query, k=TOP_K, fetch_k=FETCH_K):
    return _rrf_merge([dense_search(query, fetch_k), bm25_search(query, fetch_k)], k)

def rerank(query, candidates, k=TOP_K):
    scores = reranker.predict([(query, d.page_content) for d in candidates])
    order = sorted(range(len(candidates)), key=lambda i: -scores[i])
    return [candidates[i] for i in order[:k]]

def hybrid_rerank_search(query, k=TOP_K, fetch_k=FETCH_K):
    return rerank(query, hybrid_search(query, k=fetch_k, fetch_k=fetch_k), k)

print("base 파이프라인 준비 완료 (Hybrid + Rerank)")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
BM25 토큰화: 100%|██████████| 2240/2240 [00:04<00:00, 494.45it/s] 


base 파이프라인 준비 완료 (Hybrid + Rerank)


---
## 3. Multi-Query 정의

원 질문을 LLM으로 3개 표현으로 변형한다. 변형 질문은 **원 질문과 같은 언어**로 생성해, 검색 대상 언어가 의도치 않게 바뀌는 것을 막는다. 각 질문으로 Hybrid 검색 후 RRF로 융합하고, **재정렬 기준은 원 질문**으로 삼는다(변형은 회수 폭을 넓히는 용도이므로).

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

query_llm = ChatOpenAI(model=QUERY_GEN_MODEL, temperature=0)

MQ_PROMPT = ChatPromptTemplate.from_template(
    """당신은 의료 정보 검색을 돕는 질문 변형기입니다.
아래 [원 질문]과 같은 의미이되 표현·관점이 다른 검색 질의 {n}개를 만드세요.

규칙:
- 반드시 원 질문과 같은 언어로 작성합니다.
- 문서에 쓰일 법한 용어(의학 용어, 동의어, 상위/하위 개념)를 활용합니다.
- 한 줄에 하나씩, 번호나 기호 없이 질문만 출력합니다.

[원 질문]
{question}""")

def generate_queries(question, n=N_VARIANTS):
    out = (query_llm | StrOutputParser()).invoke(MQ_PROMPT.format(question=question, n=n))
    variants = [ln.strip() for ln in out.split("\n") if ln.strip()][:n]
    return [question] + variants

def multiquery_search(question, k=TOP_K, fetch_k=FETCH_K):
    queries = generate_queries(question)
    doc_lists = [hybrid_search(q, k=fetch_k, fetch_k=fetch_k) for q in queries]
    fused = _rrf_merge(doc_lists, k=fetch_k)
    return rerank(question, fused, k)   # 재정렬 기준은 원 질문

---
## 4. 변형 질문 확인

채점 전에 LLM이 만든 변형 질문이 타당한지, 그리고 검색 결과가 실제로 넓어지는지 확인한다. 5-2 Error Case였던 두 문항으로 점검한다.

In [5]:
test_queries = [
    "What are the four BI-RADS breast density categories?",   # 5-2 Error Case (CP 0.00)
    "유방암 검진은 몇 살부터 받는 것이 권장되나요?",              # 연령대별 권고가 분산된 문항
]
for q in test_queries:
    print("=" * 70)
    print(f"원 질문: {q}\n")
    variants = generate_queries(q)
    print("[생성된 변형]")
    for i, v in enumerate(variants):
        tag = "(원본)" if i == 0 else f"(변형{i})"
        print(f"  {tag} {v}")

    base_docs = hybrid_rerank_search(q)
    mq_docs = multiquery_search(q)
    base_keys = {_key(d) for d in base_docs}
    print("\n[Hybrid+Rerank top-5]")
    for i, d in enumerate(base_docs, 1):
        print(f"  {i}. [{d.metadata.get('filename','?')[:20]} p.{d.metadata.get('page','?')}] "
              f"{d.page_content.strip()[:70].replace(chr(10),' ')}")
    print("\n[Multi-Query top-5]")
    for i, d in enumerate(mq_docs, 1):
        new = "  <- 신규 회수" if _key(d) not in base_keys else ""
        print(f"  {i}. [{d.metadata.get('filename','?')[:20]} p.{d.metadata.get('page','?')}] "
              f"{d.page_content.strip()[:70].replace(chr(10),' ')}{new}")
    print()

원 질문: What are the four BI-RADS breast density categories?

[생성된 변형]
  (원본) What are the four BI-RADS breast density categories?
  (변형1) 유방 밀도 분류에서 BI-RADS의 네 가지 범주는 무엇인가요?
  (변형2) BI-RADS에서 정의하는 유방 밀도의 네 가지 카테고리는 어떤 것들이 있습니까?
  (변형3) 유방 밀도를 평가하는 BI-RADS의 네 가지 분류는 무엇인지 알려주세요.


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



[Hybrid+Rerank top-5]
  1. [nccn_breast_cancer_s p.17] see how the cancer is responding to treatment. BI-RADS breast density 
  2. [nccn_breast_cancer_s p.15] A radiologist will categorize your mammogram  results using a BI-RADS 
  3. [ncc_breast_cancer_sc p.54] ㆍPPV3: 6.5% (3/46)in BI-RADS 4 ㆍPPV2: 5.6% (3/54) in BI-RADS 3-4 ㆍPPV3
  4. [ncc_breast_cancer_sc p.53] BI-RADS category 4, and 5 lesions on mammography and US) × 100. (출처: M
  5. [ncc_breast_cancer_sc p.54] 핵심질문별 근거내용 51 표 23. 진단정확성 : 유방촬영술 후 추가검사로서 유방초음파 저자  (년도)  Country No.

[Multi-Query top-5]
  1. [nccn_breast_cancer_s p.17] see how the cancer is responding to treatment. BI-RADS breast density 
  2. [nccn_breast_cancer_s p.15] A radiologist will categorize your mammogram  results using a BI-RADS 
  3. [ncc_breast_cancer_sc p.54] ㆍPPV3: 6.5% (3/46)in BI-RADS 4 ㆍPPV2: 5.6% (3/54) in BI-RADS 3-4 ㆍPPV3
  4. [ncc_breast_cancer_sc p.53] BI-RADS category 4, and 5 lesions on mammography and US) × 100. (출처: M
  5. [ncc_breast_can

---
## 5. R5 채점 → 5구성 비교

In [6]:
import pandas as pd

llm = ChatOpenAI(model=GEN_MODEL, temperature=0)
RAG_PROMPT = ChatPromptTemplate.from_template(
    """당신은 유방암 정보 검색 보조 시스템입니다.

아래 [참고 문서]만 사용해서 [질문]에 답변하세요. 문서에 없는 내용은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 답하세요.
답변 마지막에는 반드시 다음 두 가지를 포함하세요:
1. 출처: 참고한 문서명과 페이지 (예: 출처: 국립암센터 유방암 검진 권고안, p.5)
2. 면책 문구: "이 답변은 일반 정보 제공 목적이며, 실제 진단·치료는 반드시 의료진과 상의하세요."

[참고 문서]
{context}

[질문]
{question}

[답변]""")

def format_context(docs):
    return "\n\n---\n\n".join(
        f"[{i}] 출처: {d.metadata.get('org','?')} / {d.metadata.get('title','?')} / p.{d.metadata.get('page','?')}\n{d.page_content}"
        for i, d in enumerate(docs, 1))

golden = pd.read_csv(DATA_EVAL / "golden_set_v1.csv").to_dict("records")
print(f"golden_set: {len(golden)}문항")

golden_set: 30문항


In [7]:
import nest_asyncio; nest_asyncio.apply()
from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model=JUDGE_MODEL, temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(embeddings)
METRICS = [faithfulness, answer_relevancy, context_precision]
METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision"]

rows = []
for g in tqdm(golden, desc="RAG[R5_multiquery]"):
    docs = multiquery_search(g["question"])
    ans = (llm | StrOutputParser()).invoke(
        RAG_PROMPT.format(context=format_context(docs), question=g["question"]))
    rows.append({"user_input": g["question"], "response": ans,
                 "retrieved_contexts": [d.page_content for d in docs],
                 "reference": g.get("ground_truth", "")})
ds = EvaluationDataset.from_list(rows)
df_r5 = evaluate(dataset=ds, metrics=METRICS, llm=ragas_llm, embeddings=ragas_emb).to_pandas()
df_r5.to_csv(DATA_PROCESSED / "week5_ragas_R5_multiquery.csv", index=False, encoding="utf-8-sig")
print("R5 완료")

RAG[R5_multiquery]: 100%|██████████| 30/30 [03:44<00:00,  7.47s/it]


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

R5 완료


In [8]:
score_tables = {
    "R1_dense": pd.read_csv(DATA_PROCESSED / "week5_ragas_R1_dense.csv"),
    "R2_bm25": pd.read_csv(DATA_PROCESSED / "week5_ragas_R2_bm25.csv"),
    "R3_hybrid": pd.read_csv(DATA_PROCESSED / "week5_ragas_R3_hybrid.csv"),
    "R4_hybrid_rerank": pd.read_csv(DATA_PROCESSED / "week5_ragas_R4_hybrid_rerank.csv"),
    "R5_multiquery": df_r5,
}

def _lang(q): return "KO" if re.search("[가-힣]", str(q)) else "EN"

rows = []
for name, df in score_tables.items():
    row = {"retriever": name}
    for c in METRIC_COLS:
        row[c] = round(df[c].mean(), 4)
    rows.append(row)
cmp = pd.DataFrame(rows)
for c in METRIC_COLS:
    cmp[c + "_delta"] = (cmp[c] - cmp[c].iloc[0]).round(4)
cmp.to_csv(DATA_PROCESSED / "week5_final_comparison.csv", index=False, encoding="utf-8-sig")
print(cmp.to_string())

print("\n--- 언어별 context_precision ---")
for L in ["KO", "EN"]:
    for name, df in score_tables.items():
        d2 = df.copy(); d2["_l"] = d2["user_input"].apply(_lang)
        print(f"  [{L}] {name:18s} CP={d2[d2['_l']==L]['context_precision'].mean():.4f}")
    print()

          retriever  faithfulness  answer_relevancy  context_precision  faithfulness_delta  answer_relevancy_delta  context_precision_delta
0          R1_dense        0.7487            0.6120             0.9072              0.0000                  0.0000                   0.0000
1           R2_bm25        0.6542            0.4482             0.8579             -0.0945                 -0.1638                  -0.0493
2         R3_hybrid        0.7784            0.6090             0.9378              0.0297                 -0.0030                   0.0306
3  R4_hybrid_rerank        0.8028            0.6105             0.9331              0.0541                 -0.0015                   0.0259
4     R5_multiquery        0.7746            0.6260             0.9464              0.0259                  0.0140                   0.0392

--- 언어별 context_precision ---
  [KO] R1_dense           CP=0.9602
  [KO] R2_bm25            CP=0.9040
  [KO] R3_hybrid          CP=0.9883
  [KO] R4_hybrid_rera

---
## 6. 문항별 분석 — Multi-Query가 무엇을 바꿨나

In [9]:
r4 = score_tables["R4_hybrid_rerank"][["user_input", "context_precision", "faithfulness"]]
r5 = df_r5[["user_input", "context_precision", "faithfulness"]]
m = r4.merge(r5, on="user_input", suffixes=("_R4", "_R5"))
m["lang"] = m["user_input"].apply(_lang)
m["cp_d"] = (m["context_precision_R5"] - m["context_precision_R4"]).round(3)
m["f_d"] = (m["faithfulness_R5"] - m["faithfulness_R4"]).round(3)

print("=== R5 vs R4: CP 변화 (|delta| > 0.05) ===")
for _, r in m[abs(m["cp_d"]) > 0.05].sort_values("cp_d", ascending=False).iterrows():
    print(f"  {r['cp_d']:+.2f} ({r['lang']}) R4={r['context_precision_R4']:.2f} -> R5={r['context_precision_R5']:.2f} | {str(r['user_input'])[:42]}")
print(f"\nCP  개선 {(m['cp_d']>0.05).sum()} / 악화 {(m['cp_d']<-0.05).sum()} / 동일 {(abs(m['cp_d'])<=0.05).sum()}")
print(f"Faith 개선 {(m['f_d']>0.05).sum()} / 악화 {(m['f_d']<-0.05).sum()}")

print("\n=== 5-2 Error Case 3문항 추적 ===")
for kw in ["BI-RADS", "몇 살부터", "increased risk"]:
    row = m[m["user_input"].str.contains(kw, na=False, case=False)]
    for _, r in row.iterrows():
        print(f"  {kw}: CP {r['context_precision_R4']:.2f} -> {r['context_precision_R5']:.2f} | "
              f"Faith {r['faithfulness_R4']:.2f} -> {r['faithfulness_R5']:.2f}")

=== R5 vs R4: CP 변화 (|delta| > 0.05) ===
  +0.32 (KO) R4=0.68 -> R5=1.00 | 유방암 검진은 몇 살부터 받는 것이 권장되나요?
  +0.15 (EN) R4=0.80 -> R5=0.95 | What factors put a person at increased ris
  +0.13 (EN) R4=0.87 -> R5=1.00 | Which risk assessment models are commonly 
  +0.11 (EN) R4=0.89 -> R5=1.00 | When might a breast MRI be recommended for
  -0.08 (EN) R4=1.00 -> R5=0.92 | At what age should annual screening mammog
  -0.11 (KO) R4=1.00 -> R5=0.89 | 유방암 1기와 2기의 차이는 무엇인가요?
  -0.17 (EN) R4=0.80 -> R5=0.64 | What are the different types of breast bio

CP  개선 4 / 악화 3 / 동일 23
Faith 개선 5 / 악화 8

=== 5-2 Error Case 3문항 추적 ===
  BI-RADS: CP 1.00 -> 1.00 | Faith 0.83 -> 0.82
  BI-RADS: CP 0.00 -> 0.00 | Faith 0.71 -> 0.62
  몇 살부터: CP 0.68 -> 1.00 | Faith 0.67 -> 1.00
  increased risk: CP 0.80 -> 0.95 | Faith 1.00 -> 0.85


---
## 판정 & 기록

- **Multi-Query 효과 (수치)**: R4 대비 CP +0.013(0.9331→0.9464, 5구성 중 최고), AR +0.016, **Faithfulness −0.028**(0.8028→0.7746). 문항 단위로는 CP 개선 4 / 악화 3, Faith 개선 5 / **악화 8**.

- **5-2 Error Case 해소 여부**:
  - **검진 연령**: CP 0.68→1.00, Faith 0.67→1.00 — **해소.** 변형 질문("적정 연령은 몇 세인가요", "언제부터 시행하는 것이 좋습니까")이 base에 없던 chunk(kbcs p.198, p.215 — 가족력 기반 조기검진)를 신규 회수. "분산된 근거를 넓게 회수한다"는 가설이 검증된 사례.
  - **위험 요인**: CP 0.80→0.95 — **해소.** R4는 가족력·유전적 소인 2개만 답하고 AR 0.00이었으나, R5는 방사선 노출·연령·비만·음주 등 8개 항목과 위험 배수까지 서술.
  - **BI-RADS 밀도**: CP 0.00→0.00 — **미해소.** 변형 질문은 정상 생성됐으나 top-5가 base와 완전히 동일하고 신규 회수 없음. 검색된 chunk는 모두 판독 범주(0~6)이고 질문이 묻는 밀도 범주(A~D)가 아니다. **문서에 해당 내용이 인덱싱되지 않은 데이터 커버리지 문제**이며, 질문 변형으로는 해결 불가.

- **정성 검증 — 수치와 어긋난 사례**: 답변 텍스트를 직접 비교한 결과, 지표 우위와 실제 유용성이 일치하지 않았다.
  - **식이요법(결정적)**: R5는 **CP 1.00으로 검색은 최고였으나 답변은 "제공된 문서에서 확인할 수 없습니다"**(Faith 0.00, AR 0.00). R4는 채소·과일 섭취, 짜지 않게, 탄 음식 회피, 식사일기, 고열량 식품 보관 등 4개 항목을 구체적으로 답변. **관련 chunk를 찾고도 답을 생성하지 못한 기능 실패**로, CP가 "답변 생성 가능성"을 보장하지 않음을 보여준다.
  - **생검 유형**: R4는 4종(FNA·코어·진공보조·절제 생검) + 마취·이미징 안내, R5는 3종으로 **절제 생검 누락**. 회수 폭을 넓혔음에도 항목이 줄어듦.
  - **1기·2기**: 대등. R4는 IIB까지 서술, R5는 IIB를 "문서에 포함되어 있지 않다"고 정직하게 표기하고 결론을 간결화. Faith는 R5가 낮으나(모른다는 진술이 근거 없는 문장으로 감점), 종양 크기만으로 병기를 단정하지 않은 점은 도메인 관점에서 더 안전한 서술.

- **구현상 발견 — 언어 누출**: 영어 질문("What are the four BI-RADS breast density categories?")에 대해 변형 질문 3개가 모두 한국어로 생성됨. 프롬프트에 "원 질문과 같은 언어" 제약을 명시했으나 지켜지지 않았다. 다국어 corpus에서 질문 변형 시 검색 대상 언어가 의도치 않게 바뀔 수 있는 함정.

- **트레이드오프**: 질문당 LLM 호출 1회 + 검색 4회 + 재정렬로 지연이 크게 증가. 변형 질문이 매번 새로 생성되므로 결과 재현성도 낮아진다.

- **판정**: **Multi-Query 미채택. 5주차 최종 retrieval = Hybrid + Rerank (R4).**
  집계 CP는 R5가 소폭 높으나, ① 식이요법에서 답변 생성 자체가 실패했고 ② 나열형 질문에서 항목이 누락됐으며 ③ 언어 누출 결함이 확인됐고 ④ 지연 증가 폭이 크다. 정성 비교 4건에서 R4 2승 / R5 1승 / 1무.
  다만 **분산형 질문(검진 연령·위험 요인)에서는 명확히 유효**했으므로, 전면 적용이 아닌 **질문 유형별 선택적 적용**이 방향이다.

- **잔여 실패 케이스 (6주차 Agentic RAG 대상)**:
  1. **BI-RADS 밀도** — 재검색으로 해결 불가(문서 부재). **답변 거절(refusal)** 로직과 데이터 커버리지 점검 대상.
  2. **질문 유형별 최적 전략 상이** — 현재는 모든 질문에 동일 전략을 적용한다. "이 질문에 재검색·질문 변형이 필요한가"를 판단하는 노드가 필요하다는 근거.
  3. **CP 고득점-답변 실패 사례(식이요법)** — 검색 품질을 평가하는 grade 노드가 "이 chunk로 답변 가능한가"까지 판단해야 함을 시사.

- decision_log(Week 5) 추가: "Multi-Query 미채택 — 집계 CP는 소폭 높으나 식이요법 문항에서 답변 생성 실패, 생검 유형 항목 누락, 영어 질문의 한국어 변형 누출 확인. 분산형 질문에는 유효해 선택적 적용을 향후 과제로 남김."